# 8. Train ModernBERT Prefilter

This notebook trains the paragraph prefilter used before LLM database/source extraction. It expects a manually labelled paragraph sheet with at least `text` and `label` columns. Labels `1` and `2` are collapsed into the positive class.


## Train Binary Prefilter And Compare Thresholds


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

#determine ideal threshold - nr of paragraphs passed setting

import pandas as pd
import numpy as np
import torch
from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# =========================================================
# SETTINGS
# =========================================================
MODEL_NAME = "answerdotai/ModernBERT-base"

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.
TRAIN_FILE = DATA_ROOT / "training_data" / "paragraph_coding_HT.xlsx"
OUTPUT_DIR = DATA_ROOT / "training_data" / "modernbert_db_prefilter_binary_finetuned"

MAX_LENGTH = 512
RANDOM_STATE = 42
CHOSEN_THRESHOLD = 0.30  # change after inspecting validation threshold results

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_excel(TRAIN_FILE)

df = df[["text", "label"]].dropna().copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""]
df["label"] = df["label"].astype(int)

# Collapse labels: 0 stays 0; 1 and 2 become 1
df["label"] = df["label"].replace({2: 1})

print("\nCollapsed label distribution:")
print(df["label"].value_counts(dropna=False).sort_index())

# =========================================================
# TRAIN / VALIDATION / TEST SPLIT
# =========================================================
# For 500 rows: 300 train, 100 validation, 100 test
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=train_val_df["label"]
)

print("\nTrain distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTest distribution:")
print(test_df["label"].value_counts().sort_index())

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

# =========================================================
# TOKENIZER
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)
test_ds = test_ds.map(tokenize_function, batched=True)

keep_cols = ["input_ids", "attention_mask", "label"]

train_ds = train_ds.remove_columns(
    [col for col in train_ds.column_names if col not in keep_cols]
)
val_ds = val_ds.remove_columns(
    [col for col in val_ds.column_names if col not in keep_cols]
)
test_ds = test_ds.remove_columns(
    [col for col in test_ds.column_names if col not in keep_cols]
)

# =========================================================
# MODEL
# =========================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# =========================================================
# METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, labels=[0, 1], average=None, zero_division=0
    )

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_0": precision_per_class[0],
        "recall_0": recall_per_class[0],
        "f1_0": f1_per_class[0],
        "precision_1": precision_per_class[1],
        "recall_1": recall_per_class[1],
        "f1_1": f1_per_class[1],
    }

# =========================================================
# TRAINING
# =========================================================
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="recall_1",
    greater_is_better=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=2,
    report_to="none",
    dataloader_pin_memory=False,
    disable_tqdm=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

# =========================================================
# VALIDATION METRICS WITH ARGMAX
# =========================================================
val_metrics = trainer.evaluate(val_ds)

print("\nFinal validation metrics using argmax:")
for k, v in val_metrics.items():
    print(f"{k}: {v}")

# =========================================================
# THRESHOLD COMPARISON ON VALIDATION SET
# =========================================================
val_pred_output = trainer.predict(val_ds)

val_logits = val_pred_output.predictions
val_labels = val_pred_output.label_ids

val_probs = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
val_prob_positive = val_probs[:, 1]

thresholds = np.arange(0.1, 1.0, 0.1) 

threshold_results = []

for threshold in thresholds:
    val_preds = (val_prob_positive >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels,
        val_preds,
        average="binary",
        zero_division=0
    )

    nr_passed = val_preds.sum()
    share_passed = val_preds.mean()

    threshold_results.append({
        "threshold": round(threshold, 2),
        "nr_passed_to_llm": int(nr_passed),
        "share_passed_to_llm": share_passed,
        "precision_1": precision,
        "recall_1": recall,
        "f1_1": f1,
    })

threshold_df = pd.DataFrame(threshold_results)

# Clean, publication-ready version for the supplementary materials
threshold_table = (
    threshold_df
    .rename(columns={
        "threshold": "Decision threshold",
        "nr_passed_to_llm": "Paragraphs retained (n)",
        "share_passed_to_llm": "Paragraphs retained (%)",
        "precision_1": "Precision",
        "recall_1": "Recall",
        "f1_1": "F1 score",
    })
    [[
        "Decision threshold",
        "Paragraphs retained (n)",
        "Paragraphs retained (%)",
        "Precision",
        "Recall",
        "F1 score",
    ]].copy()
)

threshold_table["Paragraphs retained (%)"] *= 100
threshold_table = threshold_table.round({
    "Decision threshold": 1,
    "Paragraphs retained (%)": 1,
    "Precision": 3,
    "Recall": 3,
    "F1 score": 3,
})

# LaTeX/booktabs-style vector table for display and publication export
display_table = pd.DataFrame({
    "Decision threshold": threshold_table["Decision threshold"].map(lambda value: f"{value:.1f}"),
    "Paragraphs retained (n)": threshold_table["Paragraphs retained (n)"].map(lambda value: f"{value:,.0f}"),
    "Paragraphs retained (%)": threshold_table["Paragraphs retained (%)"].map(lambda value: f"{value:.1f}"),
    "Precision": threshold_table["Precision"].map(lambda value: f"{value:.3f}"),
    "Recall": threshold_table["Recall"].map(lambda value: f"{value:.3f}"),
    "F1 score": threshold_table["F1 score"].map(lambda value: f"{value:.3f}"),
})

table_headers = [
    ["Decision", "threshold"],
    ["Paragraphs retained", "(n)"],
    ["Paragraphs retained", "(%)"],
    ["Precision"],
    ["Recall"],
    ["F1 score"],
]
column_widths = [170, 230, 250, 150, 150, 150]
left_margin = 50
svg_width = left_margin * 2 + sum(column_widths)
svg_height = 490

column_centers = []
current_x = left_margin
for width in column_widths:
    column_centers.append(current_x + width / 2)
    current_x += width
right_edge = current_x

svg_parts = [
    f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {svg_width} {svg_height}" width="{svg_width}" height="{svg_height}">',
    '<rect width="100%" height="100%" fill="white"/>',
    '<desc>ModernBERT validation performance across decision thresholds</desc>',
    f'<text x="{left_margin}" y="31" font-family="STIX Two Text, Times New Roman, serif" font-size="21" font-weight="600">ModernBERT performance across decision thresholds on the validation set</text>',
    f'<line x1="{left_margin}" y1="53" x2="{right_edge}" y2="53" stroke="black" stroke-width="2.2"/>',
]

for center, header_lines in zip(column_centers, table_headers):
    header_y_positions = [86] if len(header_lines) == 1 else [77, 97]
    for header_text, header_y in zip(header_lines, header_y_positions):
        svg_parts.append(
            f'<text x="{center:.1f}" y="{header_y}" text-anchor="middle" '
            'font-family="STIX Two Text, Times New Roman, serif" font-size="16" '
            f'font-weight="600">{header_text}</text>'
        )

svg_parts.append(
    f'<line x1="{left_margin}" y1="113" x2="{right_edge}" y2="113" stroke="black" stroke-width="1.2"/>'
)

row_y_start = 143
row_step = 38
for row_index, (_, row) in enumerate(display_table.iterrows()):
    row_y = row_y_start + row_index * row_step
    is_selected = np.isclose(
        threshold_table.iloc[row_index]["Decision threshold"],
        CHOSEN_THRESHOLD,
    )
    font_weight = "700" if is_selected else "400"
    for center, column in zip(column_centers, display_table.columns):
        svg_parts.append(
            f'<text x="{center:.1f}" y="{row_y}" text-anchor="middle" '
            'font-family="STIX Two Text, Times New Roman, serif" font-size="16" '
            f'font-weight="{font_weight}">{row[column]}</text>'
        )

bottom_rule_y = row_y_start + (len(display_table) - 1) * row_step + 23
svg_parts.extend([
    f'<line x1="{left_margin}" y1="{bottom_rule_y}" x2="{right_edge}" y2="{bottom_rule_y}" stroke="black" stroke-width="2.2"/>',
    '</svg>',
])

svg_content = "\n".join(svg_parts)

# High-resolution PNG preview of the same table
from PIL import Image, ImageDraw, ImageFont

scale = 2
png_image = Image.new("RGB", (svg_width * scale, svg_height * scale), "white")
draw = ImageDraw.Draw(png_image)
font_dir = Path(r"C:\Windows\Fonts")
regular_font = ImageFont.truetype(str(font_dir / "times.ttf"), 16 * scale)
bold_font = ImageFont.truetype(str(font_dir / "timesbd.ttf"), 16 * scale)
title_font = ImageFont.truetype(str(font_dir / "timesbd.ttf"), 21 * scale)
draw.text(
    (left_margin * scale, 31 * scale),
    "ModernBERT performance across decision thresholds on the validation set",
    font=title_font,
    fill="black",
    anchor="ls",
)
draw.line(
    (left_margin * scale, 53 * scale, right_edge * scale, 53 * scale),
    fill="black",
    width=4,
)

for center, header_lines in zip(column_centers, table_headers):
    header_y_positions = [86] if len(header_lines) == 1 else [77, 97]
    for header_text, header_y in zip(header_lines, header_y_positions):
        draw.text(
            (center * scale, header_y * scale),
            header_text,
            font=bold_font,
            fill="black",
            anchor="mm",
        )

draw.line(
    (left_margin * scale, 113 * scale, right_edge * scale, 113 * scale),
    fill="black",
    width=2,
)

for row_index, (_, row) in enumerate(display_table.iterrows()):
    row_y = row_y_start + row_index * row_step
    is_selected = np.isclose(
        threshold_table.iloc[row_index]["Decision threshold"],
        CHOSEN_THRESHOLD,
    )
    row_font = bold_font if is_selected else regular_font
    for center, column in zip(column_centers, display_table.columns):
        draw.text(
            (center * scale, row_y * scale),
            row[column],
            font=row_font,
            fill="black",
            anchor="mm",
        )

draw.line(
    (left_margin * scale, bottom_rule_y * scale, right_edge * scale, bottom_rule_y * scale),
    fill="black",
    width=4,
)
display(png_image)

threshold_output_file = OUTPUT_DIR / "modernbert_validation_threshold_performance.csv"
threshold_table.to_csv(threshold_output_file, index=False)

svg_output_file = OUTPUT_DIR / "modernbert_validation_threshold_performance.svg"
svg_output_file.write_text(svg_content, encoding="utf-8")

png_output_file = OUTPUT_DIR / "modernbert_validation_threshold_performance.png"
png_image.save(png_output_file, dpi=(300, 300))

print(f"Saved supplementary table data to: {threshold_output_file}")
print(f"Saved vector table visualization to: {svg_output_file}")
print(f"Saved PNG table visualization to: {png_output_file}")


## Evaluate Chosen Threshold On Test Split


In [ ]:

# =========================================================
# FINAL TEST EVALUATION WITH CHOSEN THRESHOLD
# =========================================================
test_pred_output = trainer.predict(test_ds)

test_logits = test_pred_output.predictions
test_labels = test_pred_output.label_ids

test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
test_prob_positive = test_probs[:, 1]

test_preds = (test_prob_positive >= CHOSEN_THRESHOLD).astype(int)

test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    test_labels,
    test_preds,
    average="binary",
    zero_division=0
)

test_nr_passed = test_preds.sum()
test_share_passed = test_preds.mean()

print("\nFinal test performance with chosen threshold:")
print(f"threshold: {CHOSEN_THRESHOLD}")
print(f"nr_passed_to_llm: {test_nr_passed}")
print(f"share_passed_to_llm: {test_share_passed}")
print(f"precision_1: {test_precision}")
print(f"recall_1: {test_recall}")
print(f"f1_1: {test_f1}")


## Save Final Model


In [ ]:
# =========================================================
# SAVE FINAL MODEL + TOKENIZER
# =========================================================
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

## Optional: Apply Model To Unlabelled Paragraphs


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.
MODEL_DIR = DATA_ROOT / "training_data" / "modernbert_db_prefilter_binary_finetuned"
LABELLED_FILE = DATA_ROOT / "training_data" / "paragraph_coding_HT.xlsx"
THRESHOLD = 0.30   # lower threshold = higher recall

df_texts_to_process = pd.read_excel(LABELLED_FILE, index_col=0)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

def predict_paragraphs(texts, batch_size=16):
    all_results = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()

        for text, prob in zip(batch_texts, probs):
            pred = int(prob >= THRESHOLD)
            all_results.append({
                "text": text,
                "prob_relevant": float(prob),
                "pred_label": pred
            })

    return pd.DataFrame(all_results)

# example
texts = df_texts_to_process[df_texts_to_process['label'].isna()]['text'].dropna().to_list()

df_preds = predict_paragraphs(texts)
print(df_preds[:50])